# 900 · Quality, security & operations — procedural layer

**Domain mnemonic — five NINES:** 99.999% — the digit of never failing.

Codes covered: **903** arrange-act-assert · **916** property-based testing · **924** git bisect · **933** merge · **946** feature flags · **951** SQL injection · **964** password hashing · **976** percentiles vs averages · **983** batching round trips · **991** structured logging.

**Method:** read the cell → **predict the output** → run it → compare. The matrix holds the imagery; this notebook engraves the net effect. Cells run top to bottom in one kernel. No cell raises: the one deliberately failing test is caught and *reported* by the unittest runner — that report is the output to study.

## 903 · Arrange-Act-Assert

Three-part test structure: set up the starting state, invoke the one behavior under test, then verify the outcome.

*A three-act play: stagehands ARRANGE the props, the ACTor performs exactly one scene, then a critic stands up and ASSERTs the verdict.*

Watch: two tests, three beats each. One passes (`ok`), one fails on a deliberately wrong expectation — the runner reports `F` and points its traceback at the assert line. Predict which test fails and what the final tally says.

In [1]:
import unittest

def add(a, b):
    return a + b

class TestAdd(unittest.TestCase):
    def test_add_two_positives(self):
        a, b = 2, 3            # Arrange: set up the starting state
        result = add(a, b)     # Act: invoke the one behavior under test
        self.assertEqual(result, 5)   # Assert: verify the outcome

    def test_add_deliberately_wrong_expectation(self):
        a, b = 2, 2                    # Arrange
        result = add(a, b)             # Act
        self.assertEqual(result, 5)    # Assert: wrong on purpose -> runner reports F

unittest.main(argv=["nb"], exit=False, verbosity=2)
print("^ one F above is expected: the runner CAUGHT the failed assert and reported it.")
print("  Each test reads as three beats: arrange, act, assert.")

test_add_deliberately_wrong_expectation (__main__.TestAdd.test_add_deliberately_wrong_expectation) ... 

FAIL


test_add_two_positives (__main__.TestAdd.test_add_two_positives) ... 

ok


FAIL: test_add_deliberately_wrong_expectation (__main__.TestAdd.test_add_deliberately_wrong_expectation)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_27265/2378604519.py", line 15, in test_add_deliberately_wrong_expectation
    self.assertEqual(result, 5)    # Assert: wrong on purpose -> runner reports F
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: 4 != 5



----------------------------------------------------------------------
Ran 2 tests in 0.002s

FAILED (failures=1)


^ one F above is expected: the runner CAUGHT the failed assert and reported it.
  Each test reads as three beats: arrange, act, assert.


## 916 · Property-based testing

Generate hundreds of random inputs and check an invariant holds for all, shrinking any failure to a minimal counterexample.

*Instead of tasting three cookies, a factory robot bakes 10,000 random cookies and checks one law: none may leave the oven raw. It hands you the SMALLEST raw cookie it found.*

Watch: first a true invariant survives 200 random lists; then a buggy `my_abs` meets the abs-laws and a counterexample you would never hand-pick falls out. Predict which input breaks it.

In [2]:
import random

random.seed(42)

trials = 200
failures = []
for _ in range(trials):
    xs = [random.randint(-100, 100) for _ in range(random.randint(0, 15))]
    if list(reversed(list(reversed(xs)))) != xs:
        failures.append(xs)

print(f"property  rev(rev(xs)) == xs  checked on {trials} random lists")
print(f"failures: {len(failures)}  -> the invariant HELD for every input")

property  rev(rev(xs)) == xs  checked on 200 random lists
failures: 0  -> the invariant HELD for every input


In [3]:
random.seed(42)

def my_abs(x):
    if x > 0:
        return x
    if x < 0:
        return -x
    return -1          # bug: 'zero has no sign' -- wrong

counterexample = None
for i in range(200):
    x = random.randint(-5, 5)
    ok = my_abs(x) >= 0 and my_abs(x) in (x, -x)
    if not ok:
        counterexample = x
        break

if counterexample is None:
    print("property held on 200 random ints (it should NOT have -- widen the generator)")
else:
    print("property  my_abs(x) >= 0 and my_abs(x) in (x, -x)")
    print(f"FALSIFIED after {i + 1} random inputs")
    print(f"counterexample: x = {counterexample}  ->  my_abs({counterexample}) = {my_abs(counterexample)}")
    print("three hand-picked examples would likely have missed this; 200 random ones did not")

property  my_abs(x) >= 0 and my_abs(x) in (x, -x)
FALSIFIED after 34 random inputs
counterexample: x = 0  ->  my_abs(0) = -1
three hand-picked examples would likely have missed this; 200 random ones did not


## 924 · Git bisect

Binary-search history (or a failing input): test the midpoint, discard the good half, converge on the culprit in log2(n) steps.

*A librarian hunts the page where a novel turns bad: open at the middle, ask 'good or bad?', discard half. A thousand commits interrogated in ten questions.*

Watch: 32 simulated versions, bug hidden somewhere. Predict the probe sequence (16, then?) and how many questions it takes before you run.

In [4]:
BUG_INTRODUCED_AT = 21          # hidden truth the search must discover

def is_bad(version):
    return version >= BUG_INTRODUCED_AT

lo, hi = 1, 32                  # version 1 known good, version 32 known bad
print(f"searching versions {lo}..{hi}  (good..bad)\n")
steps = 0
while hi - lo > 1:
    mid = (lo + hi) // 2
    steps += 1
    verdict = "BAD " if is_bad(mid) else "good"
    print(f"step {steps}: probe v{mid:<2}  -> {verdict}   (window was lo=v{lo}, hi=v{hi})")
    if is_bad(mid):
        hi = mid
    else:
        lo = mid

print(f"\nfirst bad version: v{hi}  found in {steps} probes (log2(32) = 5)")
print("git bisect automates exactly this: bad HEAD, good v2.1, then answer good/bad per checkout")

searching versions 1..32  (good..bad)

step 1: probe v16  -> good   (window was lo=v1, hi=v32)
step 2: probe v24  -> BAD    (window was lo=v16, hi=v32)
step 3: probe v20  -> good   (window was lo=v16, hi=v24)
step 4: probe v22  -> BAD    (window was lo=v20, hi=v24)
step 5: probe v21  -> BAD    (window was lo=v20, hi=v22)

first bad version: v21  found in 5 probes (log2(32) = 5)
git bisect automates exactly this: bad HEAD, good v2.1, then answer good/bad per checkout


## 933 · Merge

Combines two branch histories with a merge commit that has both parents, preserving each branch's true history.

*Two rivers meet and flow on as one; standing at the confluence — the merge commit — you can still see both original riverbeds stretching behind you.*

Watch: this is REAL git in a throwaway temp directory (the enclosing repo is never touched). Predict the shape of `git log --graph` before and after the merge — and how many parents commit M has.

In [5]:
import subprocess, tempfile, os

tmp = tempfile.mkdtemp(prefix="merge_demo_")     # isolated repo -- never touches this project

def git(*args):
    r = subprocess.run(["git", *args], cwd=tmp, capture_output=True, text=True)
    return r.stdout.strip()

git("-c", "init.defaultBranch=main", "init", "-q")
git("config", "user.name", "Demo")
git("config", "user.email", "demo@example.com")

def commit(fname, msg):
    with open(os.path.join(tmp, fname), "w") as f:
        f.write(msg + "\n")
    git("add", fname)
    git("commit", "-q", "-m", msg)

commit("base.txt", "A: base commit on main")
git("switch", "-q", "-c", "feature")
commit("feature.txt", "B: work on feature branch")
git("switch", "-q", "main")
commit("main.txt", "C: parallel work on main")

print("two branches have diverged from A:")
print(git("log", "--graph", "--oneline", "--all"))

two branches have diverged from A:
* 25d66ea B: work on feature branch
| * d78dfa0 C: parallel work on main
|/  
* b16f68a A: base commit on main


In [6]:
git("merge", "feature", "-q", "-m", "M: merge feature into main")

print("after `git merge feature` -- the merge commit M joins both lines:")
print(git("log", "--graph", "--oneline", "--all"))
print()
print("parents of the merge commit:", git("log", "-1", "--format=%p"))
print("both riverbeds are still visible behind the confluence.")
print("`git rebase main` (on feature) would instead REWRITE B onto C: one straight line, no M.")

after `git merge feature` -- the merge commit M joins both lines:
*   77cce1e M: merge feature into main
|\  
| * 25d66ea B: work on feature branch
* | d78dfa0 C: parallel work on main
|/  
* b16f68a A: base commit on main

parents of the merge commit: d78dfa0 25d66ea
both riverbeds are still visible behind the confluence.
`git rebase main` (on feature) would instead REWRITE B onto C: one straight line, no M.


## 946 · Feature flags

Runtime switches that decouple deploy from release: ship code dark, enable per user or percentage, kill instantly on trouble.

*The new chandelier is fully wired into the ballroom but its wall switch stays OFF; you flip it for the staff first, then table by table — and flip it back the instant sparks fly.*

Watch: one function, three calls, three different behaviors — and not a single line of `checkout` changes between them. Predict all three lines.

In [7]:
FLAGS = {"new_checkout": False}       # runtime switch, no redeploy needed

def checkout(user, total):
    if FLAGS["new_checkout"]:
        return f"NEW path: {user} pays {total * 0.9:.2f} (loyalty discount, one-page flow)"
    return f"old path: {user} pays {total:.2f} (classic three-page flow)"

print("deployed code contains BOTH paths. Same build, three moments in time:")
print()
print("flag off (dark launch):", checkout("ada", 100.0))

FLAGS["new_checkout"] = True          # release: flip the switch, ship nothing
print("flag on  (released)  :", checkout("ada", 100.0))

FLAGS["new_checkout"] = False         # trouble at 2 AM? flip it back
print("flag off (rollback)  :", checkout("ada", 100.0))
print()
print("deploy put the code on the server; the FLAG released it -- and un-released it instantly.")

deployed code contains BOTH paths. Same build, three moments in time:

flag off (dark launch): old path: ada pays 100.00 (classic three-page flow)
flag on  (released)  : NEW path: ada pays 90.00 (loyalty discount, one-page flow)
flag off (rollback)  : old path: ada pays 100.00 (classic three-page flow)

deploy put the code on the server; the FLAG released it -- and un-released it instantly.


## 951 · SQL injection

Attacker-controlled text concatenated into SQL executes as SQL, reading or destroying data — e.g. input like `' OR 1=1--`.

*A visitor signs the guestbook: `Bob'); DROP TABLE guests;--` and the naive clerk reads the name aloud as a command — the entire guest list vanishes. Data spoken as code.*

Watch (defensive education): the same hostile "password" goes through two query-building styles. Predict how many rows each returns — then read the concatenated query string carefully to see why.

In [8]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (id INTEGER, name TEXT, password TEXT)")
conn.executemany("INSERT INTO users VALUES (?, ?, ?)",
                 [(1, "ada", "s3cret"), (2, "grace", "hopper!"), (3, "alan", "enigma")])

attacker_input = "' OR '1'='1"

query = f"SELECT id, name FROM users WHERE name = 'ada' AND password = '{attacker_input}'"
print("THE ATTACK -- query built by string concatenation:")
print(" ", query)
rows = conn.execute(query).fetchall()
print("  rows returned:", rows, " <- logged in as EVERYONE, no password known")
print()
print("THE FIX -- parameterized query, same hostile input:")
rows = conn.execute("SELECT id, name FROM users WHERE name = ? AND password = ?",
                    ("ada", attacker_input)).fetchall()
print("  rows returned:", rows, " <- the input stayed DATA, never became SQL")
print()
print("Never build SQL from strings. Placeholders, always.")

THE ATTACK -- query built by string concatenation:
  SELECT id, name FROM users WHERE name = 'ada' AND password = '' OR '1'='1'
  rows returned: [(1, 'ada'), (2, 'grace'), (3, 'alan')]  <- logged in as EVERYONE, no password known

THE FIX -- parameterized query, same hostile input:
  rows returned: []  <- the input stayed DATA, never became SQL

Never build SQL from strings. Placeholders, always.


## 964 · Password hashing

Store only slow, salted, one-way hashes (bcrypt, argon2) so stolen databases cannot be reversed and rainbow tables are useless.

*Each password goes through a deliberately SLOW one-way meat grinder with a pinch of unique salt, so two identical steaks grind into different mince — and nothing can be un-ground.*

Watch: the hash prefixes change on every run — salts are **supposed** to be random; that is the one intentional nondeterminism in this notebook. The verdicts (`identical? False`, `True`/`False` on verify) never change. Never store plaintext.

In [9]:
import hashlib, os

def hash_pw(password, salt, iterations=100_000):
    return hashlib.pbkdf2_hmac("sha256", password.encode(), salt, iterations)

pwd = "correct horse battery staple"
salt1 = os.urandom(16)     # salts SHOULD be random -- that is the point of a salt
salt2 = os.urandom(16)
h1 = hash_pw(pwd, salt1)
h2 = hash_pw(pwd, salt2)

print("same password, two different random salts:")
print("  salt1 -> hash", h1.hex()[:16] + "...")
print("  salt2 -> hash", h2.hex()[:16] + "...")
print("  hashes identical?", h1 == h2, " -> rainbow tables are useless")
print()

def verify(password, salt, stored_hash):
    return hash_pw(password, salt) == stored_hash    # re-hash and compare; never un-hash

print("verify(correct password):", verify(pwd, salt1, h1))
print("verify(wrong password)  :", verify("hunter2", salt1, h1))
print()
print("store (salt, hash) only -- 100,000 iterations make each guess deliberately slow")

same password, two different random salts:
  salt1 -> hash 0123d382fb40c952...
  salt2 -> hash 840775189e027ed3...
  hashes identical? False  -> rainbow tables are useless



verify(correct password): True


verify(wrong password)  : False

store (salt, hash) only -- 100,000 iterations make each guess deliberately slow


## 976 · Percentiles vs averages

Report p50/p95/p99 latency rather than the mean: averages hide the slow tail that real users actually experience.

*'This river averages one meter deep,' says the sign — straight over a nine-meter trench that soaks every tenth crosser. p50 is the median crosser; p99 nearly drowned.*

Watch: 95 requests near 10 ms plus 5 at 500 ms. Predict the mean by hand first — (95×10 + 5×500)/100 — then see which of the four numbers describes a user you have actually met.

In [10]:
import random, statistics

random.seed(7)
latencies = [random.gauss(10, 1) for _ in range(95)] + [500.0] * 5   # 95 fast, 5 terrible
random.shuffle(latencies)

q = statistics.quantiles(latencies, n=100)     # 99 cut points
mean   = statistics.mean(latencies)
median = statistics.median(latencies)
p95, p99 = q[94], q[98]

print("100 requests: 95 around 10 ms, 5 at 500 ms")
print()
print(f"  mean   : {mean:7.1f} ms   <- 'looks fine', says nobody's actual experience")
print(f"  median : {median:7.1f} ms   (p50: the typical user)")
print(f"  p95    : {p95:7.1f} ms")
print(f"  p99    : {p99:7.1f} ms   <- 1 user in 100 waits ~50x longer than the median")
print()
print("the mean averaged the pain away; the percentiles kept it. Alert on p99, not the mean.")

100 requests: 95 around 10 ms, 5 at 500 ms

  mean   :    34.4 ms   <- 'looks fine', says nobody's actual experience
  median :    10.2 ms   (p50: the typical user)
  p95    :   475.6 ms
  p99    :   500.0 ms   <- 1 user in 100 waits ~50x longer than the median

the mean averaged the pain away; the percentiles kept it. Alert on p99, not the mean.


## 983 · Batching round trips

Combine many small network or disk operations into one request, paying the fixed per-trip latency once instead of N times.

*The courier stops making a hundred round trips for a hundred parcels and loads one truck: the fixed toll per trip — the network round-trip — is paid once instead of a hundred times.*

Watch: the cost model is printed in the code (0.002 s per CALL + 0.0001 s per item). Predict both timings arithmetically before running; exact milliseconds wobble, the ~20x ratio does not.

In [11]:
import time

def fake_api(items):
    time.sleep(0.002)                  # fixed per-CALL overhead: the network round trip
    time.sleep(0.0001 * len(items))    # marginal per-item work
    return [x * 2 for x in items]

items = list(range(100))

t0 = time.perf_counter()
for x in items:
    fake_api([x])                      # 100 round trips: pay the toll 100 times
chatty = time.perf_counter() - t0

t0 = time.perf_counter()
fake_api(items)                        # 1 round trip: pay the toll once
batched = time.perf_counter() - t0

print(f"100 single-item calls : {chatty * 1000:7.1f} ms   (100 x overhead + 100 x item)")
print(f"  1 batched call(100) : {batched * 1000:7.1f} ms   (  1 x overhead + 100 x item)")
print(f"speedup               : {chatty / batched:5.1f}x")
print()
print("same work per item -- the win is paying the fixed per-trip cost once instead of N times")

100 single-item calls :   231.4 ms   (100 x overhead + 100 x item)
  1 batched call(100) :    12.3 ms   (  1 x overhead + 100 x item)
speedup               :  18.7x

same work per item -- the win is paying the fixed per-trip cost once instead of N times


## 991 · Structured logging

Emit logs as key-value events (JSON) with level, timestamp, and correlation IDs — machine-queryable, not prose strings.

*The librarian swaps diary scrawl for index cards with labeled fields — user_id: 42, order: A7, ms: 340 — and suddenly a machine finds every card where ms > 1000 in seconds.*

Watch: four events go in, then one dict-lookup filter (`user == "ada"`) pulls exactly the right three back out — including the WARNING. Predict which three. Try writing that query against prose logs.

In [12]:
import json, logging, itertools

_tick = itertools.count()

class JsonFormatter(logging.Formatter):
    def format(self, record):
        return json.dumps({
            "ts": f"2026-07-11T12:00:{next(_tick):02d}Z",   # fixed clock -> deterministic output
            "level": record.levelname,
            "event": record.getMessage(),
            "user": getattr(record, "user", None),
        })

class ListHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.lines = []
    def emit(self, record):
        self.lines.append(self.format(record))

log = logging.getLogger("shop")
log.setLevel(logging.INFO)
log.propagate = False
handler = ListHandler()
handler.setFormatter(JsonFormatter())
log.handlers = [handler]

log.info("login",            extra={"user": "ada"})
log.info("add_to_cart",      extra={"user": "grace"})
log.warning("payment_retry", extra={"user": "ada"})
log.info("logout",           extra={"user": "ada"})

print("raw log lines (one JSON object per event):")
for line in handler.lines:
    print(" ", line)

events = [json.loads(line) for line in handler.lines]      # machine-parseable
ada_events = [e for e in events if e["user"] == "ada"]
print()
print(f"query: user == 'ada'  ->  {len(ada_events)} events")
for e in ada_events:
    print(f"  {e['ts']}  {e['level']:<7}  {e['event']}")
print()
print("try that filter against prose like 'Ada logged in OK!!' -- keys beat grep-and-pray")

raw log lines (one JSON object per event):
  {"ts": "2026-07-11T12:00:00Z", "level": "INFO", "event": "login", "user": "ada"}
  {"ts": "2026-07-11T12:00:01Z", "level": "INFO", "event": "add_to_cart", "user": "grace"}
  {"ts": "2026-07-11T12:00:02Z", "level": "WARNING", "event": "payment_retry", "user": "ada"}
  {"ts": "2026-07-11T12:00:03Z", "level": "INFO", "event": "logout", "user": "ada"}

query: user == 'ada'  ->  3 events
  2026-07-11T12:00:00Z  INFO     login
  2026-07-11T12:00:02Z  WARNING  payment_retry
  2026-07-11T12:00:03Z  INFO     logout

try that filter against prose like 'Ada logged in OK!!' -- keys beat grep-and-pray
